In [ ]:
import pandas as pd

base_df = pd.read_csv("../results/base_model_results.csv")
fine_tuned_df = pd.read_csv("../results/fine_tuned_results.csv")

fine_tuned_output = fine_tuned_df[["id", "finetuned_model_output"]]

combined_df = base_df.merge(
    fine_tuned_output,
    on="id",
    how="inner"
)

combined_df = combined_df[
    [
        "id",
        "question",
        "expected_output",
        "base_model_output",
        "finetuned_model_output"
    ]
]

combined_df.to_csv(
    "../results/evaluation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Broj primjera: {len(combined_df)}")

In [ ]:
import pandas as pd

# Učitavanje postojeće tablice
df = pd.read_csv("../results/evaluation_results.csv")

# Dodavanje praznih stupaca
new_columns = [
    "base_coverage",
    "base_correctness",
    "base_reasoning",
    "base_unsupported_information",
    "base_clarity",
    "base_overall",
    "ft_coverage",
    "ft_correctness",
    "ft_reasoning",
    "ft_unsupported_information",
    "ft_clarity",
    "ft_overall",
    "winner",
    "comment"
]

for column in new_columns:
    df[column] = pd.NA

# Spremanje
df.to_csv(
    "../results/evaluation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Broj primjera: {len(df)}")

In [ ]:
import json
import time
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

client = OpenAI(api_key="YOUR API KEY")

INPUT_CSV = "../results/evaluation_results.csv"
OUTPUT_CSV = "../results/evaluation_results.csv"

SYSTEM_PROMPT = """
You are an expert evaluator of auditing language models.

You are given:

1. A question.
2. A reference answer.
3. A base model answer.
4. A fine-tuned model answer.

The reference answer is the ground truth.

Evaluate BOTH answers independently.

Do NOT compare them while assigning scores.

Use the following metrics (0-100):

Coverage:
How completely the answer covers the reference answer.

Correctness:
How factually correct the answer is.

Reasoning:
How well the answer explains WHY, not only WHAT.

Unsupported_information:
100 = introduces no unsupported information.
0 = introduces substantial unsupported information.

Clarity:
Writing quality, organization and readability.

Overall:
Overall quality considering all previous criteria.

After scoring both answers, indicate which answer is better.

Return ONLY valid JSON.

Example:

{
  "base": {
    "coverage":85,
    "correctness":89,
    "reasoning":80,
    "unsupported_information":59,
    "clarity":56,
    "overall":62
  },
  "fine_tuned":{
    "coverage":67,
    "correctness":84,
    "reasoning":94,
    "unsupported_information":89,
    "clarity":80,
    "overall":69
  },
  "winner":"fine_tuned",
  "comment":"Fine-tuned answer stays closer to the reference answer."
}
"""

df = pd.read_csv(INPUT_CSV)
df["winner"] = df["winner"].astype("string")
df["comment"] = df["comment"].astype("string")

# Dodaj stupce ako ne postoje
columns = [
    "base_coverage",
    "base_correctness",
    "base_reasoning",
    "base_unsupported_information",
    "base_clarity",
    "base_overall",
    "ft_coverage",
    "ft_correctness",
    "ft_reasoning",
    "ft_unsupported_information",
    "ft_clarity",
    "ft_overall",
    "winner",
    "comment"
]

for c in columns:
    if c not in df.columns:
        df[c] = None

for i, row in tqdm(df.iterrows(), total=len(df)):

    # preskoči već ocijenjene
    if pd.notna(row["winner"]):
        continue

    user_prompt = f"""
QUESTION

{row['question']}


REFERENCE ANSWER

{row['expected_output']}


BASE MODEL ANSWER

{row['base_model_output']}


FINE-TUNED ANSWER

{row['finetuned_model_output']}
"""

    success = False

    while not success:

        try:

            response = client.responses.create(

                model="gpt-5-mini",

                input=[
                    {
                        "role":"system",
                        "content":SYSTEM_PROMPT
                    },
                    {
                        "role":"user",
                        "content":user_prompt
                    }
                ],

                max_output_tokens=3000
            )

            result = json.loads(response.output_text)

            df.loc[i,"base_coverage"] = result["base"]["coverage"]
            df.loc[i,"base_correctness"] = result["base"]["correctness"]
            df.loc[i,"base_reasoning"] = result["base"]["reasoning"]
            df.loc[i,"base_unsupported_information"] = result["base"]["unsupported_information"]
            df.loc[i,"base_clarity"] = result["base"]["clarity"]
            df.loc[i,"base_overall"] = result["base"]["overall"]

            df.loc[i,"ft_coverage"] = result["fine_tuned"]["coverage"]
            df.loc[i,"ft_correctness"] = result["fine_tuned"]["correctness"]
            df.loc[i,"ft_reasoning"] = result["fine_tuned"]["reasoning"]
            df.loc[i,"ft_unsupported_information"] = result["fine_tuned"]["unsupported_information"]
            df.loc[i,"ft_clarity"] = result["fine_tuned"]["clarity"]
            df.loc[i,"ft_overall"] = result["fine_tuned"]["overall"]

            df.loc[i,"winner"] = result["winner"]
            df.loc[i,"comment"] = result["comment"]

            df.to_csv(OUTPUT_CSV,index=False)

            success = True

        except Exception as e:

            print(e)
            print("Retrying...")
            time.sleep(5)

print("Finished.")